<a href="https://colab.research.google.com/github/Yatiajitsingh/Distressed-Asset-Risk-Model/blob/main/Agritech-updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install kaggle

# 1. Upload your kaggle.json file to Colab
from google.colab import files
files.upload()

# 2. Move the kaggle.json file to the correct hidden directory
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 3. Download the 10-Year Agmarknet Dataset
!kaggle datasets download -d ishankat/daily-wholesale-commodity-prices-india-mandis

# 4. Unzip the data
!unzip daily-wholesale-commodity-prices-india-mandis.zip

import pandas as pd
import pandas as pd

# 5. Load it into Pandas to verify (Updated filename)
df_historical = pd.read_csv('commodity_price.csv')
print(df_historical.head())

Saving kaggle.json to kaggle (1).json
Dataset URL: https://www.kaggle.com/datasets/ishankat/daily-wholesale-commodity-prices-india-mandis
License(s): CC0-1.0
daily-wholesale-commodity-prices-india-mandis.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  daily-wholesale-commodity-prices-india-mandis.zip
replace commodity_price.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: commodity_price.csv     
     State District                  Market                 Commodity  \
0  Gujarat   Amreli                Damnagar         Coriander(Leaves)   
1  Gujarat   Amreli             Savarkundla  Bengal Gram(Gram)(Whole)   
2  Gujarat   Amreli             Savarkundla               Castor Seed   
3  Gujarat    Anand  Khambhat(Grain Market)       Paddy(Dhan)(Common)   
4  Gujarat    Anand  Khambhat(Grain Market)       Paddy(Dhan)(Common)   

           Variety    Grade Arrival_Date  Min_x0020_Price  Max_x0020_Price  \
0        Coriander      F

In [1]:
import requests
import pandas as pd

# We are pulling 10 years of daily climate data for the target region
url = "https://power.larc.nasa.gov/api/temporal/daily/point"

params = {
    'parameters': 'T2M_MAX,T2M_MIN,CDD0', # Max Temp, Min Temp, Cooling Degree Days
    'community': 'AG',
    'longitude': 80.2707,
    'latitude': 13.0827,
    'start': '20160101',
    'end': '20251231',
    'format': 'JSON'
}

print("Fetching 10 years of climate data...")
response = requests.get(url, params=params)
climate_data = response.json()

# Convert the raw data into a clean Pandas table
climate_df = pd.DataFrame(climate_data['properties']['parameter'])
climate_df.index = pd.to_datetime(climate_df.index, format='%Y%m%d')

print("\nSuccess! Here are the first 5 days of climate data:")
print(climate_df.head())


Fetching 10 years of climate data...

Success! Here are the first 5 days of climate data:
            T2M_MAX  T2M_MIN   CDD0
2016-01-01    27.50    20.75  24.12
2016-01-02    27.61    20.19  23.90
2016-01-03    27.74    20.18  23.96
2016-01-04    27.76    21.01  24.39
2016-01-05    27.71    20.50  24.11


In [2]:
# 1. Download the real, 24-year historical dataset
!kaggle datasets download -d vandeetshah/india-commodity-wise-mandi-dataset

# 2. Unzip the massive folder (we use -q to quietly unzip the 325 files without spamming your screen)
!unzip -q india-commodity-wise-mandi-dataset.zip

import pandas as pd

# 3. Load the specific CSV for a high-yield crop (e.g., Tomato.csv, Onion.csv, or Potato.csv)
df_real = pd.read_csv('Tomato.csv')

print("✅ Successfully loaded REAL historical data!")
print(f"Total Rows: {len(df_real)}")
print(df_real.head())


Dataset URL: https://www.kaggle.com/datasets/vandeetshah/india-commodity-wise-mandi-dataset
License(s): apache-2.0
100% 404M/404M [00:06<00:00, 68.1MB/s]

✅ Successfully loaded REAL historical data!
Total Rows: 2338352
          State Name District Name Market Name Variety       Group  \
0  Arunachal Pradesh   West Kameng     Bomdila  Tomato  Vegetables   
1  Arunachal Pradesh        Tawang      Tawang   Other  Vegetables   
2  Arunachal Pradesh        Tawang      Tawang   Other  Vegetables   
3  Arunachal Pradesh        Tawang      Tawang   Other  Vegetables   
4  Arunachal Pradesh        Tawang      Tawang   Other  Vegetables   

   Arrivals (Tonnes)  Min Price (Rs./Quintal)  Max Price (Rs./Quintal)  \
0               2.98                      0.0                      0.0   
1               0.23                   1500.0                   1800.0   
2               0.20                   1200.0                   2000.0   
3               0.23                   1200.0                   

In [3]:
import pandas as pd

# 1. Define your target states (Hyderabad is in Telangana)
target_states = ['Telangana', 'Uttar Pradesh', 'Maharashtra', 'Gujarat', 'Tamil Nadu']

# 2. Filter the massive dataset for only these specific states
regional_df = df_real[df_real['State Name'].isin(target_states)].copy()

# 3. Convert the text dates into proper Python datetime objects
regional_df['Date'] = pd.to_datetime(regional_df['Reported Date'], format='%d %b %Y', errors='coerce')

# 4. Group by Date, State, AND District (City) to get city-level daily averages
# This is the crucial step for Panel Data
city_prices = regional_df.groupby(['Date', 'State Name', 'District Name'])['Modal Price (Rs./Quintal)'].mean().reset_index()

# 5. Clean the column names for SQL merging later
city_prices.rename(columns={'Modal Price (Rs./Quintal)': 'Modal_Price'}, inplace=True)

# 6. Sort chronologically so each city's timeline is perfectly ordered
city_prices = city_prices.sort_values(['State Name', 'District Name', 'Date'])

print("✅ Multi-state city data successfully aggregated!")
print(f"Total City-Level Daily Records: {len(city_prices)}")

# Let's peek at the specific data for Hyderabad to verify
hyderabad_data = city_prices[city_prices['District Name'] == 'Hyderabad']
print(f"\nTotal Records for Hyderabad: {len(hyderabad_data)}")
print(hyderabad_data.tail())

✅ Multi-state city data successfully aggregated!
Total City-Level Daily Records: 433225

Total Records for Hyderabad: 5060
             Date State Name District Name  Modal_Price
432763 2024-01-28  Telangana     Hyderabad  1300.000000
432854 2024-01-29  Telangana     Hyderabad  1400.000000
432955 2024-01-30  Telangana     Hyderabad  1366.666667
433057 2024-01-31  Telangana     Hyderabad  1520.000000
433156 2024-02-01  Telangana     Hyderabad  1620.000000


In [4]:
import requests
import pandas as pd
import time

# 1. Define coordinates for our 5 flagship cities
city_coords = {
    'Hyderabad': {'lat': 17.3850, 'lon': 78.4867},
    'Lucknow': {'lat': 26.8467, 'lon': 80.9462},
    'Pune': {'lat': 18.5204, 'lon': 73.8567},
    'Ahmedabad': {'lat': 23.0225, 'lon': 72.5714},
    'Chennai': {'lat': 13.0827, 'lon': 80.2707}
}

climate_frames = []
url = "https://power.larc.nasa.gov/api/temporal/daily/point"

print("Initiating satellite data extraction for target cities...")

# 2. Loop through each city to pull its specific historical weather
for city, coords in city_coords.items():
    print(f"Pulling climate data for {city}...")
    params = {
        'parameters': 'T2M_MAX,T2M_MIN,CDD0',
        'community': 'AG',
        'longitude': coords['lon'],
        'latitude': coords['lat'],
        'start': '20160101',
        'end': '20240201', # Aligning with our market data end date
        'format': 'JSON'
    }

    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        df = pd.DataFrame(data['properties']['parameter'])
        df.index = pd.to_datetime(df.index, format='%Y%m%d')
        df.reset_index(inplace=True)
        df.rename(columns={'index': 'Date'}, inplace=True)
        df['District Name'] = city # Tag the city for merging
        climate_frames.append(df)

    time.sleep(1) # Brief pause to prevent API timeout

# 3. Combine all city weather into one massive dataset
master_climate_df = pd.concat(climate_frames, ignore_index=True)

# 4. Filter your market dataset for only these 5 cities
pilot_cities = list(city_coords.keys())
filtered_market_df = city_prices[city_prices['District Name'].isin(pilot_cities)]

# 5. THE MERGE: Join market prices and climate data on BOTH Date and City
final_master_db = pd.merge(filtered_market_df, master_climate_df, on=['Date', 'District Name'], how='inner')

# 6. Clean up missing days using forward-fill
final_master_db['Modal_Price'] = final_master_db.groupby('District Name')['Modal_Price'].ffill()
final_master_db = final_master_db.dropna()

print("\n✅ ETL Pipeline Complete: Master Relational Database Generated!")
print(f"Total Unified Records: {len(final_master_db)}")
print(final_master_db.head())


Initiating satellite data extraction for target cities...
Pulling climate data for Hyderabad...
Pulling climate data for Lucknow...
Pulling climate data for Pune...
Pulling climate data for Ahmedabad...
Pulling climate data for Chennai...

✅ ETL Pipeline Complete: Master Relational Database Generated!
Total Unified Records: 10054
        Date State Name District Name  Modal_Price  T2M_MAX  T2M_MIN   CDD0
0 2016-01-01    Gujarat     Ahmedabad       1450.0    34.35    14.65  24.50
1 2016-01-02    Gujarat     Ahmedabad       1400.0    32.88    13.69  23.29
2 2016-01-04    Gujarat     Ahmedabad       1300.0    32.78    13.97  23.38
3 2016-01-05    Gujarat     Ahmedabad       1200.0    32.67    14.51  23.59
4 2016-01-06    Gujarat     Ahmedabad       1000.0    32.62    15.96  24.29


In [5]:
import statsmodels.api as sm

# 1. Define our Independent Variables (The Shocks: Heat and Cooling Load)
X = final_master_db[['T2M_MAX', 'CDD0']]

# 2. Define our Dependent Variable (The Revenue: Market Price)
y = final_master_db['Modal_Price']

# 3. Add a constant (the intercept) to establish our baseline
X = sm.add_constant(X)

# 4. Fit the Econometric Model
print("Running Multivariable Regression Model...\n")
model = sm.OLS(y, X).fit()

# 5. Output the professional statistical summary
print(model.summary())


Running Multivariable Regression Model...

                            OLS Regression Results                            
Dep. Variable:            Modal_Price   R-squared:                       0.156
Model:                            OLS   Adj. R-squared:                  0.155
Method:                 Least Squares   F-statistic:                     926.5
Date:                Thu, 10 Sep 2026   Prob (F-statistic):               0.00
Time:                        12:42:49   Log-Likelihood:                -85082.
No. Observations:               10054   AIC:                         1.702e+05
Df Residuals:                   10051   BIC:                         1.702e+05
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const    

In [6]:
!pip install streamlit
!npm install localtunnel


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 41.9 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦
added 22 packages in 3s
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦

In [7]:
%%writefile app.py
import streamlit as st

# 1. The Website Header
st.title("Hydroponic Smart Farming System")
st.header("Financial Risk & Bankruptcy Calculator")
st.write("Use the sliders below to simulate a heatwave and see if the farm survives.")

# 2. The Interactive Sliders
st.subheader("Set the Weather Conditions")
temperature = st.slider("Maximum Outside Temperature (°C)", 30.0, 50.0, 35.0)
ac_effort = st.slider("Air Conditioning Load (Cooling Degree Days)", 10.0, 30.0, 20.0)
electricity_bill = st.slider("Daily Electricity Bill (₹)", 500, 5000, 2000)

# 3. The Math (Using your exact regression numbers!)
baseline_revenue = 2232.43
heat_penalty = (temperature - 30) * -200.36 # Drops revenue as it gets hotter
ac_reward = (ac_effort - 10) * 222.78       # Adds revenue for surviving

total_revenue = baseline_revenue + heat_penalty + ac_reward
daily_profit = total_revenue - electricity_bill

# 4. The Final Verdict (The Psychology of Risk)
st.subheader("The Financial Reality Check")

col1, col2 = st.columns(2)
col1.metric("Expected Market Revenue", f"₹ {int(total_revenue)}")
col2.metric("Daily Profit / Loss", f"₹ {int(daily_profit)}")

if daily_profit < 0:
    st.error("🚨 BANKRUPTCY RISK: The electricity bill is destroying your profits. The market does not pay enough to cover this heatwave!")
else:
    st.success("✅ PROFITABLE: Your farm is making money under these conditions.")

Writing app.py


In [9]:
import urllib
print("Password/Enpoint IP for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))

!streamlit run app.py & npx localtunnel --port 8501

Password/Enpoint IP for localtunnel is: 34.73.19.117
⠙

2026-09-10 13:04:49.934 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.73.19.117:8501

your url is: https://lazy-cycles-notice.loca.lt
  Stopping...
^C
